In [1]:
# Install required libraries
!pip install -qU langchain langchain-community langchain-google-genai huggingface_hub langchain-text-splitters pypdf sentence-transformers faiss-cpu pandas numpy langchain-huggingface

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.8/161.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/

In [ ]:
# Restart runtime after install (Colab sometimes needs this for clean imports)
import os
os.kill(os.getpid(), 9)

In [1]:
# Imports
import os
import time
import numpy as np
import pandas as pd
from pathlib import Path

from IPython.display import display

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI

/tmp/ipykernel_1622/1845955896.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
# Upload PDF file
from google.colab import files

uploaded = files.upload()
pdf_file = Path(next(iter(uploaded.keys())))

print("Uploaded PDF:", pdf_file.name)

Saving Institutional-Distribution-in-Computer-Science.pdf to Institutional-Distribution-in-Computer-Science.pdf
Uploaded PDF: Institutional-Distribution-in-Computer-Science.pdf


In [3]:
# Load PDF
loader = PyPDFLoader(str(pdf_file))
documents = loader.load()

print(f"Loaded {len(documents)} pages")

Loaded 6 pages


In [4]:
# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"Split into {len(chunks)} chunks")
print("\nSample chunk:\n", chunks[0].page_content[:300])

Split into 20 chunks

Sample chunk:
 Annals of Library and Information Studies 49,1; 2002; 23-27 
Institutional distribution in computer science research in India: a study/ Anup Kumar Das and Aruna Karanjai, 2002. 
 
23 
INSTITUTIONAL DISTRIBUTION IN COMPUTER SCIENCE RESEARCH IN INDIA: A STUDY 
/G03 
ANUP KUMAR DAS 
Email: anupdas2072@


In [5]:
# Create embeddings model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings model loaded successfully.


In [6]:
# Create FAISS vector database from chunks + embeddings
vector_store = FAISS.from_documents(chunks, embeddings)

print("FAISS vector database created successfully.")
print("Total vectors stored:", vector_store.index.ntotal)

FAISS vector database created successfully.
Total vectors stored: 20


In [7]:
# Set retrieval depth
TOP_K = 4

# Wrap the FAISS vector store into a retriever
retriever = vector_store.as_retriever(
    search_kwargs={"k": TOP_K}
)

print("Retriever ready.")
print("Retrieval depth (TOP_K):", TOP_K)

Retriever ready.
Retrieval depth (TOP_K): 4


In [8]:
from transformers import pipeline

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_NEW_TOKENS = 200

generator = pipeline(
    "text-generation",
    model=MODEL_NAME,
    do_sample=False  # greedy decoding -> deterministic, reproducible answers
)

# Avoid conflict between model's default max_length and our max_new_tokens
generator.model.generation_config.max_length = None
generator.model.generation_config.max_new_tokens = MAX_NEW_TOKENS

print("Language model loaded successfully.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Language model loaded successfully.


In [10]:
def ask(question: str) -> None:
    """
    Runs one full RAG cycle for a single question:
      - semantic retrieval of relevant chunks
      - grounded prompt construction
      - LLM generation
      - printed answer + source citations
    """

    # --- Retrieval: embed question, find TOP_K closest chunks by meaning ---
    results = retriever.invoke(question)

    if not results:
        print("\nNo relevant information found.")
        return

    # --- Build context block with source labels ---
    context_parts = []
    for i, document in enumerate(results, start=1):
        page = document.metadata.get("page", "Unknown")
        if page != "Unknown":
            page = page + 1  # LangChain pages are 0-indexed
        text = document.page_content
        context_parts.append(f"[Source {i} - Page {page}]\n{text}")

    context = "\n\n".join(context_parts)

    # --- Grounded prompt: forbids outside knowledge, gives a safe fallback ---
    prompt = f"""You are a research paper question answering assistant.

Answer the question using ONLY the information provided in the context.
Do not use outside knowledge. Do not invent information.
If the answer is not available in the context, say:
I could not find this information in the paper.
Give a short and clear answer.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:"""

    # --- Generate ---
    output = generator(prompt, return_full_text=False)
    answer = output[0]["generated_text"]

    # --- Display answer ---
    print("\n--------------------------------------------")
    print("ANSWER")
    print("--------------------------------------------")
    print(answer.strip())

    # --- Display source citations (deduplicated by page) ---
    print("\n--------------------------------------------")
    print("SOURCE CITATION")
    print("--------------------------------------------")

    used_pages = set()
    for document in results:
        page = document.metadata.get("page", "Unknown")
        page_display = (page + 1) if page != "Unknown" else "Unknown"
        if page_display not in used_pages:
            print(f"- {pdf_file.name}, Page {page_display}")
            used_pages.add(page_display)

    print("--------------------------------------------")

In [11]:
STANDARD_QUESTIONS = {
    "Objective": "What is the objective of the paper?",
    "Methodology": "What methodology was used in this paper?",
    "Datasets": "What datasets were used in this paper?",
    "Findings": "What are the major findings of this paper?",
    "Limitations": "What are the limitations of this paper?",
}

print("\n============================================")
print(" STANDARD RESEARCH PAPER ANALYSIS ")
print("============================================")

for label, standard_question in STANDARD_QUESTIONS.items():
    print(f"\n>>> {label}: {standard_question}")
    ask(standard_question)

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 STANDARD RESEARCH PAPER ANALYSIS 

>>> Objective: What is the objective of the paper?


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--------------------------------------------
ANSWER
--------------------------------------------
The objective of the paper is to study the institutional distribution in computer science research in India and its impact on the country's potential for conducting computer science research of international standard. 

This conclusion is drawn from the abstract provided in Source 2, where it states "The study is based on 1408 research papers published in the international journals on computer sciences contributed by the Indian scientists from 1991 to 2000." This indicates that the primary focus of the paper is to analyze the contributions made by Indian scientists to computer science research over a specific period, specifically focusing on the year 2000. The paper aims to understand how these contributions align with the country's goals of conducting international-standard research and to identify any patterns or trends in the institutional distribution of research in India. It also seek

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--------------------------------------------
ANSWER
--------------------------------------------
Data pertaining to 1991 to 2000 was downloaded from the annual CD versions of the Science Citations Index (SCI). The search from SCI was executed through Boolean searching procedure, which is described below:

♦ At first, address search was carried out by the term India. (Set 1)
♦ Then the searches by the journal names were carried out. For example, Fuzzy Sets and Systems.
♦ Finally, all the journal names were combined by the operator OR. Limited journal names could be taken for a single set. For example, Fuzzy Sets and Systems OR Computers & Mathematics OR ……. (Set 2).

This method involved searching for terms related to India in the SCI database, then combining the search results using the operator OR, followed by limiting the search to specific journal names or sets of journal names that met certain criteria. This approach allowed researchers to identify and compile data from various In

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--------------------------------------------
ANSWER
--------------------------------------------
The dataset used in this paper includes:

1. Research papers
2. Institutional distribution in computer science research in India
3. Journal names
4. Cited references
5. Address of contributors
6. Title of publications
7. Source of publication
8. Document type
9. Summarized and cumulative data
10. Rearranged data from MS-FoxPro database files into MS-Excel worksheets
11. Data extracted from MS-FoxPro database files
12. Summarized and cumulative data from MS-FoxPro database files
13. Reorganized data from MS-FoxPro database files into MS-Excel worksheets
14. Data extracted from MS-FoxPro database files
15. Summarized and cumulative data from MS-FoxPro database files
16. Reorganized data from MS-FoxPro database files into MS-Excel worksheets
17. Data extracted from MS-FoxPro database files
18. Summarized and cumulative data from MS-FoxPro database files
19. Reorganized data from MS-FoxPro dat

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--------------------------------------------
ANSWER
--------------------------------------------
Based on the information provided in the context, the major findings of the paper include:

1. Institutional distribution in computer science research in India: A study
2. The study was conducted over a period of 1408 research papers published between 1991 and 2000
3. The institution's contribution to the research was analyzed, showing that a few institutions had a significant impact
4. The paper found that the majority of contributions came from a small number of institutions, such as IITs (Kolkata)
5. The study also noted that there was a trend towards increasing institutionalization in computer science research in India
6. The paper concluded that while some institutions played a crucial role in the field, the overall influence of the Indian scientific community was limited compared to other developing countries
7. The study highlighted the importance of maintaining a balance between ac